# Wave 0B Colab Pro End-to-End Verification (5-Cell Stateless Runner)
1. Mount Drive & pull Git repo
2. Stage workspace & verify out-of-panel `CASP3_HUMAN` + `GFP` data
3. Zero-shot `masked-marginals` calibration (`8M` & `650M`)
4. Extract `fp16` diff vectors + short/long LoRA-LLR timing & atomic Drive sync
5. `finally:` sync, sleep 20s, `runtime.unassign()`

In [ ]:
# Cell 1: Mount Drive, clone/pull repo, install minimal GPU deps, check GPU
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO_URL = 'https://github.com/wl2116641973-dev/esm2-fitness-ood.git'
REPO_DIR = '/content/esm2-fitness-ood'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

subprocess.run(['pip', 'install', '-q', 'transformers', 'peft', 'accelerate', 'pyarrow', 'scipy'], check=True)
subprocess.run(['pip', 'install', '-q', '--no-deps', '-e', REPO_DIR], check=True)
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# Cell 2: Configure staging & Drive paths and fetch ProteinGym CV CSVs if missing
import json, pathlib, shutil, zipfile, urllib.request
import pandas as pd

REPO_DIR = pathlib.Path('/content/esm2-fitness-ood')
WORK_DIR = pathlib.Path('/content/work')
DRIVE_DIR = pathlib.Path('/content/drive/MyDrive/esm2_fitness_ood')
CV_DIR = WORK_DIR / 'cv_folds'
WORK_DIR.mkdir(parents=True, exist_ok=True)
CV_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

zip_path = WORK_DIR / 'cv_folds_singles_substitutions.zip'
if not zip_path.exists():
    urllib.request.urlretrieve(
        'https://marks.hms.harvard.edu/proteingym/ProteinGym_v1.3/cv_folds_singles_substitutions.zip',
        zip_path,
    )
with open(REPO_DIR / 'configs' / 'panel_10.json') as f:
    targets = {p['dms_id'] for p in json.load(f)['proteins']} | {
        'CASP3_HUMAN_Roychowdhury_2020', 'OXDA_RHOTO_Vanella_2023_expression'
    }
with zipfile.ZipFile(zip_path) as zf:
    for m in zf.namelist():
        stem = pathlib.Path(m).stem
        if stem in targets and m.endswith('.csv'):
            (CV_DIR / f'{stem}.csv').write_bytes(zf.read(m))
print(f'Staged {len(list(CV_DIR.glob("*.csv")))} CSVs in {CV_DIR}')

In [ ]:
# Cell 3: Zero-shot masked-marginals calibration on CASP3_HUMAN (dev) & GFP (smoke)
import pandas as pd
from esm2_ood.colab_gpu import compute_esm2_masked_marginals
from esm2_ood.io_utils import atomic_copy_and_verify
from esm2_ood.metrics import compute_spearman
from esm2_ood.splits import load_panel_dms

ref_url = 'https://raw.githubusercontent.com/OATML-Markslab/ProteinGym/main/reference_files/DMS_substitutions.csv'
ref_df = pd.read_csv(ref_url).set_index('DMS_id')

calib_results = []
for dms_id in ('CASP3_HUMAN_Roychowdhury_2020', 'GFP_AEQVI_Sarkisyan_2016'):
    df = load_panel_dms(dms_id, CV_DIR)
    wt_seq = str(ref_df.loc[dms_id, 'target_seq'])
    for size in ('8M', '650M'):
        zs = compute_esm2_masked_marginals(df, wt_seq=wt_seq, model_name_or_size=size, batch_size=32)
        rho = compute_spearman(df['DMS_score'].to_numpy(), zs)
        calib_results.append({'dms_id': dms_id, 'model': size, 'spearman': rho})
        df[f'zs_{size}'] = zs
    out_pq = WORK_DIR / f'{dms_id}_zeroshot.parquet'
    df[['mutant', 'DMS_score', 'zs_8M', 'zs_650M']].to_parquet(out_pq, index=False)
    atomic_copy_and_verify(out_pq, DRIVE_DIR / 'zeroshot' / out_pq.name)

print(pd.DataFrame(calib_results).to_string(index=False))

In [ ]:
# Cell 4: Extract fp16 diff cache & benchmark 1 short (GFP) + 1 long (OXDA) LoRA-LLR fold
import numpy as np, pandas as pd
from esm2_ood.colab_gpu import extract_frozen_diff_embeddings, train_eval_esm2_lora_llr
from esm2_ood.io_utils import atomic_copy_and_verify
from esm2_ood.splits import get_outer_split, load_panel_dms, sample_budget_split

bench_rows = []
for dms_id in ('GFP_AEQVI_Sarkisyan_2016', 'OXDA_RHOTO_Vanella_2023_activity'):
    df = load_panel_dms(dms_id, CV_DIR)
    wt_seq = str(ref_df.loc[dms_id, 'target_seq'])

    # Extract & atomically archive fp16 frozen diff vectors
    diffs = extract_frozen_diff_embeddings(df, wt_seq=wt_seq, model_name_or_size='650M', batch_size=16)
    npz_path = WORK_DIR / f'{dms_id}_650M_diff_fp16.npz'
    np.savez_compressed(npz_path, mutant=df['mutant'].to_numpy(), diffs=diffs)
    atomic_copy_and_verify(npz_path, DRIVE_DIR / 'embeddings' / npz_path.name)

    # Benchmark 1 Contiguous fold (N=256) with LoRA-LLR
    tr_pool, va_pool, te_df = get_outer_split(df, 'contiguous', test_fold=0)
    sp = sample_budget_split(tr_pool, va_pool, te_df, n_total=256, seed=42)
    res = train_eval_esm2_lora_llr(
        sp['train_df'], sp['val_df'], sp['test_df'], wt_seq=wt_seq, model_name_or_size='650M'
    )
    bench_rows.append({
        'dms_id': dms_id, 'seq_len': len(wt_seq),
        'test_spearman': res['test_spearman'], 'best_epoch': res['best_epoch'],
        'elapsed_sec': round(res['elapsed_sec'], 2),
    })

bench_csv = WORK_DIR / 'wave0b_timing_benchmark.csv'
pd.DataFrame(bench_rows).to_csv(bench_csv, index=False)
atomic_copy_and_verify(bench_csv, DRIVE_DIR / bench_csv.name)
print(pd.DataFrame(bench_rows).to_string(index=False))

In [ ]:
# Cell 5: Mandatory stateless cleanup (sync -> sleep 20s -> runtime.unassign)
import os, time
from google.colab import runtime

try:
    os.sync()
    time.sleep(20)
finally:
    runtime.unassign()